In [1]:
#| default_exp core

# core

> Minimal Bitcoin Core JSON-RPC client for blockchain inspection.

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import httpx
from pathlib import Path
from typing import Any

Bitcoind exposes a JSON-RCP server on port 8332. `httpx` is used to communicate with it.

::: {.callout-tip}
If bitcoind is running in a different machine it is useful to forward port 8332 to the machine this python code is running in:
```
  ssh -N -L 8332:127.0.0.1:8332 slashbtc@slashbtc-node
```
:::

In [4]:
#| hide
# Monkeypatch to be able to do show_doc on httpx members
import ssl, httpx._client
httpx._client.ssl = ssl

In [5]:
show_doc(httpx.Client)

---

### Client

```python

def Client(
    auth:AuthTypes | None=None, params:QueryParamTypes | None=None, headers:HeaderTypes | None=None,
    cookies:CookieTypes | None=None, verify:ssl.SSLContext | str | bool=True, cert:CertTypes | None=None,
    trust_env:bool=True, http1:bool=True, http2:bool=False, proxy:ProxyTypes | None=None,
    mounts:None | typing.Mapping[str, BaseTransport | None]=None, timeout:TimeoutTypes=Timeout(timeout=5.0),
    follow_redirects:bool=False,
    limits:Limits=Limits(max_connections=100, max_keepalive_connections=20, keepalive_expiry=5.0),
    max_redirects:int=20, event_hooks:None | typing.Mapping[str, list[EventHook]]=None, base_url:URL | str='',
    transport:BaseTransport | None=None, default_encoding:str | typing.Callable[[bytes], str]='utf-8'
)->None:


```

*An HTTP client, with connection pooling, HTTP/2, redirects, cookie persistence, etc.*

It can be shared between threads.

Usage:

```python
>>> client = httpx.Client()
>>> response = client.get('https://example.org')
```

**Parameters:**

* **auth** - *(optional)* An authentication class to use when sending
requests.
* **params** - *(optional)* Query parameters to include in request URLs, as
a string, dictionary, or sequence of two-tuples.
* **headers** - *(optional)* Dictionary of HTTP headers to include when
sending requests.
* **cookies** - *(optional)* Dictionary of Cookie items to include when
sending requests.
* **verify** - *(optional)* Either `True` to use an SSL context with the
default CA bundle, `False` to disable verification, or an instance of
`ssl.SSLContext` to use a custom context.
* **http2** - *(optional)* A boolean indicating if HTTP/2 support should be
enabled. Defaults to `False`.
* **proxy** - *(optional)* A proxy URL where all the traffic should be routed.
* **timeout** - *(optional)* The timeout configuration to use when sending
requests.
* **limits** - *(optional)* The limits configuration to use.
* **max_redirects** - *(optional)* The maximum number of redirect responses
that should be followed.
* **base_url** - *(optional)* A URL to use as the base when building
request URLs.
* **transport** - *(optional)* A transport class to use for sending requests
over the network.
* **trust_env** - *(optional)* Enables or disables usage of environment
variables for configuration.
* **default_encoding** - *(optional)* The default encoding to use for decoding
response text, if no charset information is included in a response Content-Type
header. Set to a callable for automatic character set detection. Default: "utf-8".

## RPC client

In [6]:
#| export
class BitcoinRPC:
    "Tiny synchronous JSON-RPC client for Bitcoin Core."
    def __init__(
        self,
        url: str | None = None, # Where to reach bitcoind e.g. 127.0.0.1:8332
        auth: tuple[str, str] | None = None, # Auth cookie or username+password
        timeout: float = 30.0,
    ):
        self.url, auth, timeout = url, auth, timeout
        self._client = httpx.Client(base_url=url, auth=auth, timeout=timeout)
        self._id = 0

    def call(self, method: str, *params: Any) -> Any:
        "Invoke a Bitcoin Core RPC `method` with positional `params`."
        self._id += 1
        r = self._client.post(
            "/",
            json={
                "jsonrpc": "2.0",
                "id": self._id,
                "method": method,
                "params": list(params),
            },
        )
        r.raise_for_status()
        body = r.json()
        if body.get("error"):
            raise RuntimeError(body["error"])
        return body["result"]

    def close(self):
        self._client.close()

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()

In [7]:
show_doc(BitcoinRPC.call)

---

[source](https://github.com/slashpablo/slashbtc/blob/main/slashbtc/core.py#L26){target="_blank" style="float:right; font-size:smaller"}

### BitcoinRPC.call

```python

def call(
    method:str, params:Any
)->Any:


```

*Invoke a Bitcoin Core RPC `method` with positional `params`.*

::: {.callout-note}
# Refresher on JSON-RPC (and how it differs from REST)

Both use `HTTP POST` and `JSON` BUT:

`REST` models the world as resources with `URL`s and uses `HTTP` verbs to "act" on them. Imagine:

- `GET /blocks/000000...abc`
- `GET /transactions/{txid}`
- `POST /transactions` (body = new tx)

`JSON-RPC` ignores all that. There's one endpoint `/`, always `POST`, and the body names a function to call:

```
{"jsonrpc": "2.0", "id": 1, "method": "getblock", "params": ["000000...abc"]}
{"jsonrpc": "2.0", "id": 2, "method": "sendrawtransaction", "params": ["0200..."]}
```
:::

In [8]:
url = "http://127.0.0.1:8332"
user, _, password = Path("../.secrets/bitcoin-rpc.cookie").read_text().partition(":")
auth = httpx.BasicAuth(user, password)
rpc = BitcoinRPC(url, auth)

# Exploring bitcoind RCP capabilities

bitcoind's json-rpc server includes a `help` method that is extremely valuable to understand capbabilities

In [9]:
print(rpc.call("help"))

== Blockchain ==
dumptxoutset "path"
getbestblockhash
getblock "blockhash" ( verbosity )
getblockchaininfo
getblockcount
getblockfilter "blockhash" ( "filtertype" )
getblockfrompeer "blockhash" peer_id
getblockhash height
getblockheader "blockhash" ( verbose )
getblockstats hash_or_height ( stats )
getchainstates
getchaintips
getchaintxstats ( nblocks "blockhash" )
getdeploymentinfo ( "blockhash" )
getdifficulty
getmempoolancestors "txid" ( verbose )
getmempooldescendants "txid" ( verbose )
getmempoolentry "txid"
getmempoolinfo
getrawmempool ( verbose mempool_sequence )
gettxout "txid" n ( include_mempool )
gettxoutproof ["txid",...] ( "blockhash" )
gettxoutsetinfo ( "hash_type" hash_or_height use_index )
gettxspendingprevout [{"txid":"hex","vout":n},...]
importmempool "filepath" ( options )
loadtxoutset "path"
preciousblock "blockhash"
pruneblockchain height
savemempool
scanblocks "action" ( [scanobjects,...] start_height stop_height "filtertype" options )
scantxoutset "action" ( [sca

# Use Cases

## Get most recent block's hash

In [10]:
print(rpc.call("help", "getbestblockhash"))

getbestblockhash

Returns the hash of the best (tip) block in the most-work fully-validated chain.

Result:
"hex"    (string) the block hash, hex-encoded

Examples:
> bitcoin-cli getbestblockhash 
> curl --user myusername --data-binary '{"jsonrpc": "2.0", "id": "curltest", "method": "getbestblockhash", "params": []}' -H 'content-type: application/json' http://127.0.0.1:8332/



In [11]:
rpc = BitcoinRPC(url, auth)
latest = rpc.call("getbestblockhash")
latest

'00000000000000000001ce8fdb8b33d1241792b1b11fbbc2e183c03c985f9671'

## Get most recent block details

In [12]:
print(rpc.call("help", "getblock"))

getblock "blockhash" ( verbosity )

If verbosity is 0, returns a string that is serialized, hex-encoded data for block 'hash'.
If verbosity is 1, returns an Object with information about block <hash>.
If verbosity is 2, returns an Object with information about block <hash> and information about each transaction.
If verbosity is 3, returns an Object with information about block <hash> and information about each transaction, including prevout information for inputs (only for unpruned blocks in the current best chain).

Arguments:
1. blockhash    (string, required) The block hash
2. verbosity    (numeric, optional, default=1) 0 for hex-encoded data, 1 for a JSON object, 2 for JSON object with transaction data, and 3 for JSON object with transaction data including prevout information for inputs

Result (for verbosity = 0):
"hex"    (string) A string that is serialized, hex-encoded data for block 'hash'

Result (for verbosity = 1):
{                                 (json object)
  "hash" : 

In [13]:
rpc = BitcoinRPC(url, auth)

In [14]:
block = rpc.call("getblock", latest, 0) # 0 is the verbosity level, there are 0, 1, 2 and 3
block[:25] + '...' + latest[-25:]

'00e0ff3fbbfdaabf09fe4c4b8...1b11fbbc2e183c03c985f9671'

This is an encoded version of the block's data, it is something like:

```
[80-byte block header]
[varint tx_count]
[transaction 1]
[transaction 2]
...
```

For human readability use verbosity level 1

In [15]:
block = rpc.call("getblock", latest, 1)

for k,v in block.items():
    if k == 'tx': continue
    print(f'{k}: {v}')

txns = block['tx'][:3]
for i, tx in enumerate(txns):
    print(f'tx[{i}]: {tx}')

hash: 00000000000000000001ce8fdb8b33d1241792b1b11fbbc2e183c03c985f9671
confirmations: 1
height: 947663
version: 1073733632
versionHex: 3fffe000
merkleroot: 2f28da891f83d8ee44f6a3dc413294096416b3c32a31750e62595150fcb773ac
time: 1777782179
mediantime: 1777778091
nonce: 1110117222
bits: 17021ff0
difficulty: 132472011079030.5
chainwork: 00000000000000000000000000000000000000012481334f8b9d20ac395376b0
nTx: 2501
previousblockhash: 0000000000000000000086873be53dce3cbea531631c088f4b4cfe09bfaafdbb
strippedsize: 851059
size: 1440297
weight: 3993474
tx[0]: 80859e034c3be0abba051dd755ba01fcad223b8abc7700c0e3452afc2b8ace74
tx[1]: e900d15b3398cad6f1ba0fd32684ea9dd2c5c8e8462039f8db8ff0a3ee958ed2
tx[2]: 0d55ba165815946a595e770d00e26534a219307c641f7fa31de9b8d4db042dfd


In the above we have transaction hashes, to get the details of a transaction a new call is needed

In [16]:
#| hide
import nbdev; nbdev.nbdev_export()

/Users/pablo/src/slashbtc/.venv/lib/python3.14/site-packages/nbdev/export.py:55: UserWarning: Notebook '/Users/pablo/src/slashbtc/nbs/03_transform.ipynb' uses `#| export` without `#| default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev-migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#| export` without `#| default_exp` cell.\n"
